# SPR 2026 BI-RADS — Ensemble Inference (DML Qwen2.5-14B + Qwen3-14B  ×  SALSA Qwen3-14B)
- **Model A**: Qwen2.5-14B backbone + head (DML)
- **Model B**: Qwen3-14B backbone + head (DML)
- **Model C**: Qwen3-14B CausalLM + LoRA (SALSA, 3 folds)
- Logit-space ensemble with z-score normalisation
- Hardware: 2×T4 (16 GB each)

## 1 · Install Dependencies

In [ ]:
!pip install --no-index --find-links=/kaggle/input/datasets/mohamedsadokaissaoui/bitsandbytes-0-49-2 "bitsandbytes>=0.46.1" -q
!pip install --no-index --find-links=/kaggle/input/datasets/mohamedsadokaissaoui/peft-requirements peft -q

## 2 · Imports & Environment

In [ ]:
import os
import gc
import json
import shutil
import ctypes
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from threading import Thread

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast

from transformers import AutoTokenizer, AutoModel, AutoConfig, BitsAndBytesConfig, AutoModelForCausalLM
from peft import AutoPeftModelForFeatureExtraction, PeftModel

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['CUDA_VISIBLE_DEVICES']   = '0,1'
os.environ['PYTORCH_ALLOC_CONF']     = 'expandable_segments:True'

print('GPUs available:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i} — {torch.cuda.get_device_name(i)} — {props.total_memory / 1e9:.1f} GB')

## 3 · Paths — UPDATE THESE

In [ ]:
# ── Base model paths (HF weights, read-only) ────────────────────────────────
PATH_A_BASE = '/kaggle/input/models/qwen-lm/qwen2.5/transformers/14b-instruct/1'  # Qwen2.5-14B
PATH_B_BASE = '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b/1'            # Qwen3-14B

# ── Fine-tuned adapter + head checkpoints (from DML training output) ────────
BACKBONE_A = '/kaggle/input/datasets/mohamedsadokaissaoui/dmlrun/dml_run_1/backbone_2.5_best'
HEAD_A     = '/kaggle/input/datasets/mohamedsadokaissaoui/dmlrun/dml_run_1/head_a_best.pt'

BACKBONE_B = '/kaggle/input/datasets/mohamedsadokaissaoui/dmlrun/dml_run_1/backbone_3_best'
HEAD_B     = '/kaggle/input/datasets/mohamedsadokaissaoui/dmlrun/dml_run_1/head_b_best.pt'

# ── Test CSV ─────────────────────────────────────────────────────────────────
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

# ── SALSA Qwen3-14B paths ─────────────────────────────────────────────────────
SALSA_BASE_PATH = '/kaggle/input/models/qwen-lm/qwen-3/transformers/14b/1'
SALSA_MAX_LEN   = 1024
SALSA_FOLDS     = [
    '/kaggle/input/datasets/mohamedsadokaissaoui/salsa21/salsa/backbone1',
    '/kaggle/input/datasets/mohamedsadokaissaoui/salsa21/salsa/backbone2',
    '/kaggle/input/datasets/mohamedsadokaissaoui/salsa21/salsa/backbone3',
]
SALSA_ANSWER_TOKENS = ['0', '1', '2', '3', '4', '5', '6']


## 4 · Hyper-parameters (must match training)

In [ ]:
MAX_LEN     = 896   # must match DML training MAX_LEN
NUM_CLASSES = 7
BATCH_SIZE  = 1

## 5 · Memory Utilities

In [ ]:
def clean_memory():
    gc.collect()
    ctypes.CDLL('libc.so.6').malloc_trim(0)
    torch.cuda.empty_cache()


def free_models(*models):
    """Move models to CPU, delete them, and fully flush VRAM."""
    for m in models:
        try:
            m.cpu()
        except Exception:
            pass
        del m
    gc.collect()
    ctypes.CDLL('libc.so.6').malloc_trim(0)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    for i in range(torch.cuda.device_count()):
        free_gb = torch.cuda.mem_get_info(i)[0] / 1e9
        print(f'  cuda:{i} free after cleanup: {free_gb:.2f} GB')

## 6 · Load Test Set

In [ ]:
if os.path.exists(TEST_PATH):
    test = pd.read_csv(TEST_PATH)
    print('Test shape:', test.shape)
    display(test.head())
else:
    raise FileNotFoundError(
        f'test.csv not found at {TEST_PATH}. '
        'This is expected during interactive development; '
        'it will exist in the Kaggle evaluation runtime.'
    )

assert {'ID', 'report'}.issubset(test.columns), \
    f'Unexpected columns: {test.columns.tolist()}'
print('Columns OK:', test.columns.tolist())

## 7 · Prompt Template (must match DML training prompt)

In [ ]:
BIRADS_CATEGORIES = {
    0: 'Incompleto — necessita de imagens adicionais',
    1: 'Negativo — sem achados',
    2: 'Achado benigno',
    3: 'Provavelmente benigno — seguimento em curto intervalo',
    4: 'Suspeito — biópsia deve ser considerada',
    5: 'Altamente suspeito de malignidade',
    6: 'Malignidade comprovada por biópsia',
}
CAT_STR = '\n'.join(f'    {k}: {v}' for k, v in BIRADS_CATEGORIES.items())
CLINICAL_NOTES = (
    'Regras: 0=imagem incompleta, 1=negativo, 2=benigno, '
    '3=provavelmente benigno(seguimento 6m), '
    '4=suspeito(biópsia considerar), 5=alta suspeita(biópsia obrigatória), '
    '6=malignidade confirmada por biópsia prévia.\n'
    'ATENÇÃO: 3≠2(sem seguimento), 3≠0(sem conclusão), 5≠4(alta vs suspeita).\n'
)


def create_prompt(report_text: str) -> str:
    return (
        '<|im_start|>user\n'
        'Você é um radiologista especialista em mamografia.\n\n'
        'Tarefa: Classifique o laudo mamográfico abaixo em uma das categorias BI-RADS (0–6).\n'
        'Instrução: Responda APENAS com o número da categoria. NÃO inclua explicação.\n\n'
        f'Categorias BI-RADS:\n{CAT_STR}\n\n'
        f'{CLINICAL_NOTES}\n'
        f'Laudo:\n{report_text.strip()}\n\n'
        'Categoria BI-RADS (número apenas): '
    )


test['prompt'] = test['report'].apply(create_prompt)
test['target'] = -100
print(f'Prompts created: {len(test)}')

## 8 · Tokenizers

In [ ]:
print('Loading tokenizer A (Qwen2.5-14B)...')
tokenizer_a = AutoTokenizer.from_pretrained(PATH_A_BASE, trust_remote_code=True)
tokenizer_a.padding_side = 'left'
if tokenizer_a.pad_token is None:
    tokenizer_a.pad_token = tokenizer_a.eos_token

print('Loading tokenizer B (Qwen3-14B)...')
tokenizer_b = AutoTokenizer.from_pretrained(PATH_B_BASE, trust_remote_code=True)
tokenizer_b.padding_side = 'left'
if tokenizer_b.pad_token is None:
    tokenizer_b.pad_token = tokenizer_b.eos_token

print('Tokenizer A:', tokenizer_a.__class__.__name__, '| vocab:', tokenizer_a.vocab_size)
print('Tokenizer B:', tokenizer_b.__class__.__name__, '| vocab:', tokenizer_b.vocab_size)

## 9 · Dataset & DataLoaders

We split the test set into 2 chunks — one per GPU — for each model independently.
Each model gets its own pair of dataloaders (different tokenizers, same prompts).

In [ ]:
class ReportDataset(Dataset):
    def __init__(self, prompts, targets):
        self.prompts = prompts
        self.targets = targets

    def __getitem__(self, idx):
        return self.prompts[idx], self.targets[idx]

    def __len__(self):
        return len(self.targets)


def make_split_dataloaders(prompts, targets, batch_size=BATCH_SIZE):
    """Returns list of 2 DataLoaders, one per GPU chunk."""
    length     = len(prompts)
    split_size = length // 2
    loaders    = []
    for i in range(2):
        start = i * split_size
        end   = (i + 1) * split_size if i == 0 else length
        ds    = ReportDataset(
            prompts=prompts[start:end],
            targets=targets[start:end],
        )
        dl = DataLoader(ds, batch_size=batch_size, shuffle=False, drop_last=False)
        loaders.append(dl)
        print(f'  Chunk {i}: {len(ds)} samples → cuda:{i}')
    return loaders


all_prompts = test['prompt'].to_numpy()
all_targets = test['target'].to_numpy()

print('Dataloaders for Model A:')
dataloaders_a = make_split_dataloaders(all_prompts, all_targets)
print('Dataloaders for Model B:')
dataloaders_b = make_split_dataloaders(all_prompts, all_targets)

## 10 · Adapter Config Patcher

Both adapters were saved in a Kaggle training session where the base model was at a `/tmp/...` path.
We patch `adapter_config.json` before loading so PEFT can resolve the correct base model.

In [ ]:
def patch_adapter_config(
    adapter_path: str,
    real_base_model_path: str,
    dest_dir: str = '/kaggle/working/patched_adapters',
) -> str:
    """Copy adapter to writable dir and fix the base_model_name_or_path."""
    basename = os.path.basename(adapter_path.rstrip('/'))
    new_path  = os.path.join(dest_dir, basename)

    if os.path.exists(new_path):
        shutil.rmtree(new_path)

    shutil.copytree(adapter_path, new_path)

    config_file = os.path.join(new_path, 'adapter_config.json')
    with open(config_file) as f:
        config = json.load(f)

    old_path = config.get('base_model_name_or_path', '')
    config['base_model_name_or_path'] = real_base_model_path

    with open(config_file, 'w') as f:
        json.dump(config, f, indent=4)

    print(f'  [patch] {os.path.basename(adapter_path)}')
    print(f'          {old_path} → {real_base_model_path}')
    return new_path

## 11 · Model Definitions

Two separate `Net` classes because:
- **NetA** (Qwen2.5): loaded with `AutoPeftModelForFeatureExtraction` — the adapter is self-contained
- **NetB** (Qwen3): loaded with `AutoModel` + `PeftModel.from_pretrained` — matches how DML training saved it

In [ ]:
def _bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
    )


# ── Model A: Qwen2.5-14B  (AutoPeftModelForFeatureExtraction) ────────────────
class NetA(nn.Module):
    """Qwen2.5-14B backbone (4-bit LoRA via AutoPeft) + linear head."""

    def __init__(self, base_model_path: str, adapter_path: str, device: str):
        super().__init__()
        self.config = AutoConfig.from_pretrained(base_model_path, trust_remote_code=True)

        self.backbone = AutoPeftModelForFeatureExtraction.from_pretrained(
            adapter_path,
            use_cache=False,
            torch_dtype=torch.float16,
            quantization_config=_bnb_config(),
            device_map=device,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
        self.head = nn.Linear(self.config.hidden_size, NUM_CLASSES, bias=False)

    def forward(self, encodings):
        hidden = self.backbone(**encodings).last_hidden_state[:, -1, :]
        return self.head(hidden)


# ── Model B: Qwen3-14B  (AutoModel + PeftModel.from_pretrained) ──────────────
class NetB(nn.Module):
    """Qwen3-14B backbone (4-bit LoRA via PeftModel) + linear head."""

    def __init__(self, base_model_path: str, adapter_path: str, device: str):
        super().__init__()
        self.config = AutoConfig.from_pretrained(base_model_path, trust_remote_code=True)
        self.config.use_cache = False

        base = AutoModel.from_pretrained(
            base_model_path,
            config=self.config,
            torch_dtype=torch.float16,
            quantization_config=_bnb_config(),
            device_map=device,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
        # Load LoRA adapter on top of the quantized base
        self.backbone = PeftModel.from_pretrained(
            base,
            adapter_path,
            is_trainable=False,
        )
        self.backbone.eval()

        hidden_size = self._get_hidden_size()
        self.head = nn.Linear(hidden_size, NUM_CLASSES, bias=False)

    def _get_hidden_size(self) -> int:
        cfg = self.config
        if hasattr(cfg, 'hidden_size'):
            return cfg.hidden_size
        for attr in ('text_config', 'language_model_config'):
            sub = getattr(cfg, attr, None)
            if sub and hasattr(sub, 'hidden_size'):
                return sub.hidden_size
        raise ValueError(f'Cannot find hidden_size in {type(cfg).__name__}')

    def forward(self, encodings):
        hidden = self.backbone(**encodings).last_hidden_state[:, -1, :]
        return self.head(hidden)

## 12 · Inference Function (shared)

In [ ]:
def get_preds(model, tokenizer, dataloader, device, results, result_idx):
    """Run inference on one GPU chunk and store raw logits in results[result_idx]."""
    y_pred = torch.tensor([])
    model.eval()
    with torch.no_grad():
        for batch_prompts, _ in tqdm(dataloader, desc=f'Inference {device}'):
            encodings = tokenizer(
                list(batch_prompts),
                return_tensors='pt',
                padding='max_length',
                truncation=True,
                max_length=MAX_LEN,
            ).to(device)

            with autocast(device_type='cuda', dtype=torch.float16):
                logits = model(encodings)  # [1, NUM_CLASSES]

            y_pred = torch.cat([y_pred, logits.cpu()])

    results[result_idx] = y_pred

## 13 · Per-Model Inference Runner

In [ ]:
def run_model(
    model_cls,           # NetA or NetB
    base_path: str,      # base HF model path
    adapter_path: str,   # patched adapter dir
    head_path: str,      # .pt file with head state_dict
    tokenizer,           # matching tokenizer
    dataloaders: list,   # [dl_cuda0, dl_cuda1]
    model_name: str,
) -> torch.Tensor:
    """
    Load one model across both GPUs, run parallel inference, free VRAM.

    Returns:
        raw_logits : FloatTensor [N, NUM_CLASSES] — raw logits (softmax at ensemble stage)
    """
    print(f'\n{"=" * 60}')
    print(f'  Model: {model_name}')
    print(f'  backbone : {adapter_path}')
    print(f'  head     : {head_path}')
    print(f'{"=" * 60}')

    # Load two independent copies — one per GPU
    print('  Loading on cuda:0...')
    model_0 = model_cls(base_path, adapter_path, 'cuda:0')
    print('  Loading on cuda:1...')
    model_1 = model_cls(base_path, adapter_path, 'cuda:1')

    # Load head onto each GPU
    head_state = torch.load(head_path, weights_only=True)
    model_0.head.load_state_dict(head_state)
    model_1.head.load_state_dict(head_state)
    model_0.head.to('cuda:0')
    model_1.head.to('cuda:1')

    # Parallel inference across both GPU chunks
    results = [None, None]
    t0 = Thread(target=get_preds,
                args=(model_0, tokenizer, dataloaders[0], 'cuda:0', results, 0))
    t1 = Thread(target=get_preds,
                args=(model_1, tokenizer, dataloaders[1], 'cuda:1', results, 1))
    t0.start(); t1.start()
    t0.join();  t1.join()

    # Reassemble in original row order and convert to probabilities
    logits = torch.cat([results[0], results[1]], dim=0)  # [N, NUM_CLASSES] raw logits
    print(f'  Done. Logits shape: {logits.shape}')

    # Free both GPU copies before loading the next model
    free_models(model_0, model_1)

    return logits.cpu()

## 14 · Run Inference — Model A (Qwen2.5-14B)

In [ ]:
import gc, ctypes, time

# Aggressive pre-run cleanup
gc.collect()
ctypes.CDLL('libc.so.6').malloc_trim(0)
torch.cuda.empty_cache()
torch.cuda.synchronize()
for dev in range(torch.cuda.device_count()):
    free_gb = torch.cuda.mem_get_info(dev)[0] / 1e9
    print(f'  cuda:{dev} free before Model A: {free_gb:.2f} GB')
time.sleep(3)

patched_backbone_a = patch_adapter_config(BACKBONE_A, PATH_A_BASE)

logits_a = run_model(
    model_cls    = NetA,
    base_path    = PATH_A_BASE,
    adapter_path = patched_backbone_a,
    head_path    = HEAD_A,
    tokenizer    = tokenizer_a,
    dataloaders  = dataloaders_a,
    model_name   = 'Qwen2.5-14B (Model A)',
)

## 15 · Run Inference — Model B (Qwen3-14B)

In [ ]:
# Aggressive cleanup between models
gc.collect()
ctypes.CDLL('libc.so.6').malloc_trim(0)
torch.cuda.empty_cache()
torch.cuda.synchronize()
for dev in range(torch.cuda.device_count()):
    free_gb = torch.cuda.mem_get_info(dev)[0] / 1e9
    print(f'  cuda:{dev} free before Model B: {free_gb:.2f} GB')
time.sleep(3)

patched_backbone_b = patch_adapter_config(BACKBONE_B, PATH_B_BASE)

logits_b = run_model(
    model_cls    = NetB,
    base_path    = PATH_B_BASE,
    adapter_path = patched_backbone_b,
    head_path    = HEAD_B,
    tokenizer    = tokenizer_b,
    dataloaders  = dataloaders_b,
    model_name   = 'Qwen3-14B (Model B)',
)

## 16 · Run Inference — Model C (SALSA Qwen3-14B, 3 folds)

In [ ]:
import re

# ── SALSA prompt constants (must match salsa training) ────────────────────────
# ── Must match train.py exactly ─────────────────────────────────────────────
MAX_LEN       = 1024
NUM_CLASSES   = 7
ANSWER_TOKENS = ['0', '1', '2', '3', '4', '5', '6']
ANSWER_MARKER = '<BIRADS>'

_FOLLOWUP_RE = re.compile(
    r'controle|seguimento|retorno|acompanhamento|'
    r'reavalia[çc][aã]o|bi[oó]psia|pun[cç][aã]o|'
    r'(?<!\d)(\d{1,2})\s*meses',
    re.IGNORECASE,
)
_NO_PRIOR_RE = re.compile(
    r'anteriores n[aã]o dispon[ií]veis|primeira mamografia|'
    r'sem exames anteriores|imagens indispon[ií]veis',
    re.IGNORECASE,
)
_COMPLETE_RE = re.compile(
    r'à ultrassonografia|ao ultrassom|à us |compressão localizada|'
    r'magnificação|incidências adicionais realizadas',
    re.IGNORECASE,
)

BIRADS_CATEGORIES = {
    0: 'Incompleto',
    1: 'Negativo',
    2: 'Achado benigno',
    3: 'Provavelmente benigno',
    4: 'Suspeito',
    5: 'Altamente suspeito',
    6: 'Malignidade confirmada',
}
CAT_STR = '\n'.join([f'    {k}: {v}' for k, v in BIRADS_CATEGORIES.items()])



def create_prompt_user(report_text: str) -> str:
    has_followup = bool(_FOLLOWUP_RE.search(report_text))
    has_no_prior = bool(_NO_PRIOR_RE.search(report_text))
    is_complete  = bool(_COMPLETE_RE.search(report_text))

    followup_hint = (
        '    ⚠️ Linguagem de seguimento detectada → forte indicador de categoria 3.\n'
        if has_followup else
        '    ⚠️ Sem linguagem de seguimento → se benigno, prefira categoria 2.\n'
    )
    no_prior_hint = (
        '    ⚠️ Ausência de exames anteriores NÃO implica categoria 0.\n'
        if has_no_prior else ''
    )
    complete_hint = (
        '    ⚠️ Complementação já realizada e descrita → exame COMPLETO, não use 0.\n'
        if is_complete else ''
    )

    return (
        'Você é um radiologista especialista em mamografia.\n\n'
        'Tarefa: Classifique o laudo mamográfico abaixo em BI-RADS (0–6).\n'
        'Siga os 4 passos abaixo EM ORDEM.\n\n'

        'PASSO 1 — O exame está COMPLETO?\n'
        '  → Use 0 SOMENTE se o laudo solicita imagens que AINDA NÃO foram feitas.\n'
        f'{no_prior_hint}'
        f'{complete_hint}'
        '  → Se completo, continue para o Passo 2.\n\n'

        'PASSO 2 — HÁ ALGUM ACHADO?\n'
        '  → Se não há achados → use 1 e pare.\n'
        '  → Se há achado, continue para o Passo 3.\n\n'

        'PASSO 3 — O ACHADO É BENIGNO OU SUSPEITO?\n'
        '  Se BENIGNO:\n'
        '    • Use 2 (SEM seguimento): estável ≥2 anos, linfonodo/cisto confirmado,\n'
        '      calcificações vasculares/grosseiras/em pipoca estáveis.\n'
        f'{followup_hint}'
        '    • Use 3 (COM seguimento 6m): primeiro exame + achado benigno,\n'
        '      estabilidade <12 meses, assimetria persistente sem US correlação.\n\n'
        '  Se SUSPEITO:\n'
        '    • 4 = biópsia a considerar | 5 = biópsia obrigatória | 6 = malignidade confirmada\n\n'

        'PASSO 4 — VERIFICAÇÃO: 3≠2 | 3≠0 | 5≠4\n\n'

        f'Categorias:\n{CAT_STR}\n\n'
        f'Laudo:\n{report_text.strip()}\n\n'
        f'Responda APENAS com o número BI-RADS no formato: {ANSWER_MARKER}#Número</BIRADS>\n'
        f'Onde o número é um de: 0, 1, 2, 3, 4, 5, 6'
    )


def build_inference_prompt(report_text, tok):
    messages = [
        {'role': 'user',      'content': create_prompt_user(report_text)},
        {'role': 'assistant', 'content': ANSWER_MARKER},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    if text.endswith('<|im_end|>\n'):
        text = text[:-len('<|im_end|>\n')]
    return text




print('Prompt helpers defined.')


# ── SALSANet model ────────────────────────────────────────────────────────────
class SALSANet(nn.Module):
    '''Qwen3-14B CausalLM + LoRA. No head -- SALSA reads answer token logits directly.'''

    def __init__(self, base_model_path, adapter_path, device):
        super().__init__()
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
        )
        base = AutoModelForCausalLM.from_pretrained(
            base_model_path,
            use_cache=False,
            torch_dtype=torch.float16,
            quantization_config=bnb_config,
            device_map=device,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
        self.backbone = PeftModel.from_pretrained(base, adapter_path)
        self.backbone.eval()

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return out.logits  # (B, T, vocab_size)


# ── SALSA inference (returns raw logits_7) ───────────────────────────────────
def salsa_get_preds(model, tok, dataloader, device, ans_token_ids, results, result_idx):
    '''SALSA inference: read logits of answer tokens at last position.'''
    ans_ids   = ans_token_ids.to(device)
    all_logits = []

    model.eval()
    with torch.no_grad():
        for batch_prompts in tqdm(dataloader, desc=f'Inference {device}'):
            enc = tok(
                list(batch_prompts),
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=MAX_LEN,
                add_special_tokens=False,
            ).to(device)

            with autocast(device_type='cuda', dtype=torch.float16):
                logits = model(enc['input_ids'], enc['attention_mask'])  # (B, T, V)

            # Last token -- model is primed to generate the answer digit next
            last_logits = logits[:, -1, :]                        # (B, V)
            logits_7    = last_logits.index_select(-1, ans_ids)   # (B, 7) raw
            all_logits.append(logits_7.float().cpu())  # keep raw logits

    results[result_idx] = torch.cat(all_logits, dim=0)  # (N, 7) raw logits


In [ ]:

# ── Setup SALSA tokenizer & prompts ──────────────────────────────────────────
print("Setting up SALSA tokenizer...")
salsa_tok = AutoTokenizer.from_pretrained(SALSA_BASE_PATH, trust_remote_code=True)
salsa_tok.padding_side = 'left'
if salsa_tok.pad_token is None:
    salsa_tok.pad_token = salsa_tok.eos_token

# Resolve answer token IDs
SALSA_ANSWER_TOKEN_IDS = []
for t in SALSA_ANSWER_TOKENS:
    ids = salsa_tok.encode(t, add_special_tokens=False)
    assert len(ids) == 1, f"Token '{t}' maps to multiple IDs: {ids}"
    SALSA_ANSWER_TOKEN_IDS.append(ids[0])
SALSA_ANSWER_TOKEN_IDS = torch.tensor(SALSA_ANSWER_TOKEN_IDS)
print(f'SALSA answer token IDs: {list(zip(SALSA_ANSWER_TOKENS, SALSA_ANSWER_TOKEN_IDS.tolist()))}')

# Build SALSA prompts using its own prompt function
test['salsa_prompt'] = test['report'].apply(lambda r: build_inference_prompt(r, salsa_tok))

# Split across 2 GPUs
from torch.utils.data import Dataset as _DS
class _SimpleDS(_DS):
    def __init__(self, arr): self.arr = arr
    def __len__(self): return len(self.arr)
    def __getitem__(self, i): return self.arr[i]

salsa_arr   = test['salsa_prompt'].to_numpy()
salsa_split = len(salsa_arr) // 2
salsa_dls   = [
    DataLoader(_SimpleDS(salsa_arr[:salsa_split]),  batch_size=4, shuffle=False),
    DataLoader(_SimpleDS(salsa_arr[salsa_split:]),  batch_size=4, shuffle=False),
]

# ── Run SALSA 3 folds ─────────────────────────────────────────────────────────
fold_logits_c = []
for i, fold_path in enumerate(SALSA_FOLDS):
    print(f"\n{'='*60}\n  SALSA Fold {i+1}/{len(SALSA_FOLDS)}\n{'='*60}")
    clean_memory()
    torch.cuda.synchronize()
    for dev in range(torch.cuda.device_count()):
        print(f'  cuda:{dev} free: {torch.cuda.mem_get_info(dev)[0]/1e9:.2f} GB')
    import time; time.sleep(3)

    m0 = SALSANet(SALSA_BASE_PATH, fold_path, 'cuda:0')
    m1 = SALSANet(SALSA_BASE_PATH, fold_path, 'cuda:1')

    results = [None, None]
    t0 = Thread(target=salsa_get_preds,
                args=(m0, salsa_tok, salsa_dls[0], 'cuda:0', SALSA_ANSWER_TOKEN_IDS, results, 0))
    t1 = Thread(target=salsa_get_preds,
                args=(m1, salsa_tok, salsa_dls[1], 'cuda:1', SALSA_ANSWER_TOKEN_IDS, results, 1))
    t0.start(); t1.start()
    t0.join();  t1.join()

    fold_logits = torch.cat([results[0], results[1]], dim=0)
    print(f'  Fold {i+1} logits shape: {fold_logits.shape}')
    fold_logits_c.append(fold_logits)
    free_models(m0, m1)

logits_c = sum(fold_logits_c) / len(fold_logits_c)
print(f'\nModel C (SALSA) mean logits shape: {logits_c.shape}')


## 17 · Z-score Normalise + Ensemble + Submission

In [ ]:

## 17 · Z-score Normalise + Ensemble + Submission

def zscore_logits(logits: torch.Tensor) -> torch.Tensor:
    """Per-sample z-score: removes scale/bias differences across architectures."""
    mean = logits.mean(dim=-1, keepdim=True)
    std  = logits.std(dim=-1, keepdim=True).clamp(min=1e-6)
    return (logits - mean) / std

# logits_a = raw DML logits from Qwen2.5-14B
# logits_b = raw DML logits from Qwen3-14B
# logits_c = raw SALSA logits from Qwen3-14B (different scale!)
assert logits_a.shape == logits_b.shape == logits_c.shape, \
    f'Shape mismatch: A={logits_a.shape}  B={logits_b.shape}  C={logits_c.shape}'

# Z-score normalise each model before blending
logits_a_n = zscore_logits(logits_a)
logits_b_n = zscore_logits(logits_b)
logits_c_n = zscore_logits(logits_c)
print('Z-score normalisation applied ✓')

# Weights: DML Qwen3-14B > DML Qwen2.5-14B > SALSA (1 fold each for A/B, 3 for C)
wa, wb, wc = 0.35, 0.40, 0.25
print(f'Ensemble weights — A(Qwen2.5 DML): {wa}  B(Qwen3 DML): {wb}  C(SALSA): {wc}  sum={wa+wb+wc:.2f}')

ensemble_logits = wa * logits_a_n + wb * logits_b_n + wc * logits_c_n
ensemble_probs  = F.softmax(ensemble_logits, dim=-1)
predictions     = ensemble_probs.argmax(dim=-1).numpy().clip(0, 6)

# ── Distribution comparison ───────────────────────────────────────────────────
preds_a = logits_a_n.argmax(dim=-1).numpy().clip(0, 6)
preds_b = logits_b_n.argmax(dim=-1).numpy().clip(0, 6)
preds_c = logits_c_n.argmax(dim=-1).numpy().clip(0, 6)
print('\nPrediction distribution:')
print(f'  {"Class":<12} {"A(Qwen2.5)":>12} {"B(Qwen3DML)":>12} {"C(SALSA)":>10} {"Ensemble":>10}')
for cls in range(NUM_CLASSES):
    label = BIRADS_CATEGORIES.get(cls, '')
    print(f'  BI-RADS {cls}    {int((preds_a==cls).sum()):>12} {int((preds_b==cls).sum()):>12} {int((preds_c==cls).sum()):>10} {int((predictions==cls).sum()):>10}   {label}')

n = len(predictions)
agree = int((preds_a == preds_b) & (preds_b == preds_c)).sum() if hasattr(preds_a, '__len__') else 0
import numpy as np
agree = int(np.sum((preds_a == preds_b) & (preds_b == preds_c)))
print(f'\nFull 3-model agreement: {agree}/{n} ({agree/n*100:.1f}%)')

# ── Save ──────────────────────────────────────────────────────────────────────
submission = pd.DataFrame({'ID': test['ID'].values, 'target': predictions})
submission.to_csv('submission.csv', index=False)
print('\nsubmission.csv saved!')
display(submission.head(10))
print('Shape:', submission.shape)
